# 4. Entrenamiento: Random Forest y Regresión Logística

Entrenamos dos modelos de clasificación —**Random Forest** y **Regresión Logística**— reutilizando la misma lógica de preprocesamiento de `feature_engineering.ipynb`, pero corrigiendo el punto que se dejó pendiente ahí: aquí sí separamos train/test **antes** de ajustar el `ColumnTransformer`, y lo ajustamos únicamente sobre el set de entrenamiento (dentro de un `Pipeline` junto con el clasificador).

## ¿Por qué separar antes de escalar/codificar?

`StandardScaler` calcula media y desviación estándar de los datos que ve. Si se calculan sobre train+test juntos, el modelo recibe información (aunque sea indirecta, vía esos estadísticos) del set de prueba antes de ser evaluado — eso es **data leakage**. La consecuencia es que la métrica de test queda optimista y no refleja el desempeño real ante datos nuevos. Al meter el `ColumnTransformer` dentro del `Pipeline` y llamar `pipeline.fit(X_train, y_train)`, scikit-learn garantiza que el ajuste (`fit`) del preprocesador solo ve `X_train`; `X_test` únicamente pasa por `transform` (con los estadísticos ya fijados).

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('data/titanic_clean.csv')
df.head()

,passenger_id,survived,pclass,sex,age,sibsp,parch,familiares,fare,embarked,titulo
0,1,0,3,male,22.0,1,0,1,7.2500,S,Mr
1,2,1,1,female,38.0,1,0,1,71.2833,C,Mrs
2,3,1,3,female,26.0,0,0,0,7.9250,S,Miss
3,4,1,1,female,35.0,1,0,1,53.1000,S,Mrs
4,5,0,3,male,35.0,0,0,0,8.0500,S,Mr


In [2]:
numeric_cols = ['age', 'sibsp', 'parch', 'familiares']
fare_col = ['fare']
categorical_cols = ['sex', 'embarked', 'titulo', 'pclass']
target_col = 'survived'

X = df[numeric_cols + fare_col + categorical_cols]
y = df[target_col]

y.value_counts(normalize=True)

survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

## División train/test

- **`stratify=y`**: la proporción de sobrevivientes (~38%) frente a no sobrevivientes (~62%) se conserva igual en train y en test. Sin esto, una partición aleatoria podría por azar dejar, por ejemplo, un test con 30% de sobrevivientes y un train con 40%, sesgando la evaluación. Con solo 891 filas, ese riesgo no es despreciable.
- **`random_state=42`**: fija la semilla del generador aleatorio para que la partición sea reproducible entre ejecuciones.
- **`test_size=0.2`**: es un balance estándar entre sesgo y varianza — más datos de entrenamiento reducen la varianza del modelo aprendido, pero un test más pequeño aumenta la varianza de la métrica de evaluación. Con un dataset de este tamaño, 20% (≈178 filas) sigue siendo suficiente para una estimación razonablemente estable de accuracy.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

Train: (712, 9)  Test: (179, 9)


## Preprocesador reutilizable

Definimos el mismo `ColumnTransformer` de `feature_engineering.ipynb` (escalado + `log1p` en `fare` + `OneHotEncoder(drop='first')`) dentro de una función `build_preprocessor()`. Esto es importante: cada `Pipeline` necesita **su propia instancia** del transformador. Si ambos modelos compartieran el mismo objeto `ColumnTransformer` ya construido, al hacer `.fit()` del segundo pipeline se reajustarían silenciosamente las estadísticas (medias, categorías) que el primer pipeline ya había aprendido, contaminando sus resultados aunque el primer pipeline no se vuelva a entrenar explícitamente.

In [4]:
def build_preprocessor():
    fare_pipeline = Pipeline(steps=[
        ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scale', StandardScaler())
    ])
    return ColumnTransformer(transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('fare', fare_pipeline, fare_col),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ])

## Los dos modelos

| | Random Forest | Regresión Logística |
|---|---|---|
| **Ventaja** | Captura relaciones no lineales e interacciones entre variables sin que se las tengamos que indicar a mano (p. ej. "mujer y primera clase" interactúan). Robusto a outliers. | Coeficientes interpretables directamente como log-odds; modelo simple, rápido, con pocos hiperparámetros que ajustar. |
| **Desventaja** | Menos interpretable ("caja gris"); con muchos árboles profundos puede sobreajustar. Por eso limitamos `max_depth=6`: restringe la complejidad de cada árbol para reducir varianza a costa de un poco de sesgo. | Asume una frontera de decisión lineal en el espacio de las variables transformadas; si la relación real es muy no lineal, el modelo queda con sesgo alto (underfitting). |

Usamos los mismos hiperparámetros para ambos que hiciste en el proyecto de referencia (`n_estimators=100, max_depth=6` para el bosque; `max_iter=1000` para asegurar convergencia de la regresión logística).

In [5]:
pipeline_rf = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6))
])

pipeline_lr = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

## Validación cruzada sobre el set de entrenamiento

Antes de tocar el set de test (que solo debe usarse **una vez**, al final, como estimación honesta del desempeño), revisamos qué tan estable es cada modelo con **5-fold cross-validation** sobre `X_train`. Esto nos da, además del promedio de accuracy, su desviación estándar entre folds: una forma directa de ver la varianza del estimador, algo que una sola partición train/test no puede mostrar.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in [('Random Forest', pipeline_rf), ('Regresión Logística', pipeline_lr)]:
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    print(f'{name}: accuracy CV = {scores.mean():.4f} +/- {scores.std():.4f}')

Random Forest: accuracy CV = 0.8272 +/- 0.0197
Regresión Logística: accuracy CV = 0.8216 +/- 0.0297


## Entrenamiento final y evaluación en test

In [7]:
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)

print('Random Forest')
print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print('Matriz de confusión:\n', confusion_matrix(y_test, y_pred_rf))

Random Forest
Accuracy: 0.8100558659217877
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       110
           1       0.81      0.67      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.81      0.81      0.81       179

Matriz de confusión:
 [[99 11]
 [23 46]]


In [8]:
pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

print('Regresión Logística')
print('Accuracy:', accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print('Matriz de confusión:\n', confusion_matrix(y_test, y_pred_lr))

Regresión Logística
Accuracy: 0.8324022346368715
              precision    recall  f1-score   support

           0       0.84      0.89      0.87       110
           1       0.81      0.74      0.77        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

Matriz de confusión:
 [[98 12]
 [18 51]]


**Precision vs. recall, en este problema**: precision para la clase "sobrevivió" responde "de los que predije que sobrevivían, ¿cuántos realmente sobrevivieron?"; recall responde "de los que realmente sobrevivieron, ¿a cuántos identifiqué?". No hay una respuesta única sobre cuál priorizar — depende de si el costo de un falso positivo (decir que sobrevivió y no fue así) importa más o menos que el de un falso negativo; aquí, al ser un ejercicio académico sin consecuencias reales, miramos ambas junto con `f1-score` (su media armónica) y `accuracy` en conjunto.

In [9]:
results = pd.DataFrame({
    'modelo': ['Random Forest', 'Regresión Logística'],
    'accuracy_test': [accuracy_score(y_test, y_pred_rf), accuracy_score(y_test, y_pred_lr)]
})
results

,modelo,accuracy_test
0,Random Forest,0.810056
1,Regresión Logística,0.832402


## Interpretabilidad

**Random Forest** — `feature_importances_` mide, para cada variable, cuánto reduce en promedio la impureza (Gini) al usarse en los splits de todos los árboles. **Limitación importante**: esta medida está sesgada hacia variables continuas o con muchas categorías (tienden a ofrecer más puntos de corte posibles), así que no es una medida perfecta de causalidad ni siquiera de importancia real — una alternativa más robusta (no implementada aquí) es la *permutation importance*, que mide la caída de desempeño al mezclar aleatoriamente cada columna.

In [10]:
feature_names_rf = pipeline_rf.named_steps['preprocessor'].get_feature_names_out()
importances = pd.Series(
    pipeline_rf.named_steps['classifier'].feature_importances_, index=feature_names_rf
)
importances.sort_values(ascending=False)

cat__titulo_Mr      0.205210
cat__sex_male       0.202875
fare__fare          0.138161
num__age            0.092552
cat__pclass_3       0.076952
cat__titulo_Mrs     0.061075
num__familiares     0.056042
cat__titulo_Miss    0.048198
num__sibsp          0.036672
num__parch          0.023594
cat__pclass_2       0.020290
cat__embarked_S     0.019679
cat__embarked_Q     0.009365
cat__titulo_Otro    0.009336
dtype: float64

**Regresión Logística** — cada coeficiente es el cambio en el log-odds de sobrevivir por unidad de la variable transformada (para las numéricas, por desviación estándar, ya que están escaladas; para las dummies, respecto a la categoría de referencia descartada). Aplicando `exp(coeficiente)` se obtiene el **odds ratio**: cuánto se multiplican las probabilidades relativas de sobrevivir.

In [11]:
feature_names_lr = pipeline_lr.named_steps['preprocessor'].get_feature_names_out()
coefs = pd.Series(pipeline_lr.named_steps['classifier'].coef_[0], index=feature_names_lr)
odds_ratios = np.exp(coefs)

pd.DataFrame({'coeficiente': coefs, 'odds_ratio': odds_ratios}).sort_values('coeficiente', key=abs, ascending=False)

,coeficiente,odds_ratio
cat__titulo_Mr,-2.044384,0.129460
cat__pclass_3,-1.430181,0.239266
cat__titulo_Otro,-1.001813,0.367213
cat__sex_male,-0.949856,0.386797
cat__pclass_2,-0.654680,0.519608
fare__fare,0.525472,1.691256
num__age,-0.444727,0.640999
cat__titulo_Miss,-0.426690,0.652666
cat__embarked_Q,0.351700,1.421482
num__sibsp,-0.342794,0.709784


## Guardado de los modelos

Cada `pipeline_*` guardado con `joblib` incluye el preprocesador **ya ajustado** junto con el clasificador. Esto significa que para predecir sobre un pasajero nuevo (con las columnas crudas: `sex='female'`, `embarked='C'`, etc.) basta con cargar el archivo y llamar `.predict()` — sin reescribir ninguna lógica de codificación aparte, evitando exactamente la inconsistencia manual que mencionamos en `feature_engineering.ipynb`.

In [12]:
joblib.dump(pipeline_rf, 'model_rf.joblib')
joblib.dump(pipeline_lr, 'model_lr.joblib')
print('Modelos guardados: model_rf.joblib, model_lr.joblib')

Modelos guardados: model_rf.joblib, model_lr.joblib


## Resumen

- Se entrenaron `RandomForestClassifier` y `LogisticRegression`, cada uno dentro de un `Pipeline` que incluye su propio `ColumnTransformer` ajustado únicamente sobre `X_train` (sin data leakage).
- Se validó la estabilidad de cada modelo con 5-fold cross-validation sobre train antes de evaluar (una sola vez) sobre test.
- Se revisó la interpretabilidad de ambos modelos (importancia de variables en RF, coeficientes/odds ratios en LR), señalando las limitaciones de cada medida.
- Ambos pipelines completos (preprocesamiento + modelo) se guardaron en `model_rf.joblib` y `model_lr.joblib`, listos para usarse en un futuro notebook/servicio de predicciones.